# Sección 9 — Enfoque B: $\mathcal{G}(p,c)$ cruzado por k-pairs

Implementa la fórmula definida en el MD 9:
$$\mathcal{G}(p,\,c) = \left(\bigcup_{k=1}^{4} \mathcal{G}_k(p,c)\right) \setminus \{\emptyset\}$$

El objetivo es recuperar el GUID correcto para cada uno de los **36 eventos centinela k=1**,
consultando los cuatro k-pairs en lugar de solo k=1 (Enfoque A).

## Setup

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)
pd.set_option('display.max_colwidth', 80)

NULL_GUID = '00000000-0000-0000-0000-000000000000'
DATA_DIR  = '../dataset/run-01-apt-1/'
print('OK')

In [ ]:
df = pd.read_csv(DATA_DIR + '02_sysmon-run-01.csv', low_memory=False)
df['_original_row_index'] = df.index
df['ts'] = pd.to_datetime(df['timestamp'].where(df['timestamp'] > 0), unit='ms')

viol = pd.read_csv(DATA_DIR + '04_sysmon-run-01-violations.csv', low_memory=False)
viol['ts'] = pd.to_datetime(viol['timestamp'].where(viol['timestamp'] > 0), unit='ms')

sentinel_k1 = (
    viol[~viol['EventID'].isin([8, 10]) & (viol['ProcessGuid'] == NULL_GUID)]
    .sort_values('_original_row_index')
    .reset_index(drop=True)
)

print(f'df          : {len(df):,} filas')
print(f'sentinel_k1 : {len(sentinel_k1)} eventos')

---
## Evento 1 / 36 — PID 3364 en `endofroad.boombox.local`

Primer evento centinela k=1 (fila 5976): EID=3 (NetworkConnect), `<unknown process>`,
conexión a `10.1.0.4:135` (epmap/RPC), `ts = 2025-03-19 05:01:15 UTC`.

Calculamos $\mathcal{G}_k(3364,\, \texttt{endofroad})$ para cada k-pair.

In [ ]:
c = 'endofroad.boombox.local'
p = 3364.0

# G_1: ProcessGuid donde ProcessId = p  (EID ∉ {8,10})
g1 = set(df[
    ~df['EventID'].isin([8, 10]) &
    (df['Computer'] == c) & (df['ProcessId'] == p)
]['ProcessGuid'].dropna()) - {NULL_GUID}

# G_2: ParentProcessGuid donde ParentProcessId = p  (EID = 1)
g2 = set(df[
    (df['EventID'] == 1) &
    (df['Computer'] == c) & (df['ParentProcessId'] == p)
]['ParentProcessGuid'].dropna()) - {NULL_GUID}

# G_3: SourceProcessGUID donde SourceProcessId = p  (EID ∈ {8,10})
g3 = set(df[
    df['EventID'].isin([8, 10]) &
    (df['Computer'] == c) & (df['SourceProcessId'] == p)
]['SourceProcessGUID'].dropna()) - {NULL_GUID}

# G_4: TargetProcessGUID donde TargetProcessId = p  (EID ∈ {8,10})
g4 = set(df[
    df['EventID'].isin([8, 10]) &
    (df['Computer'] == c) & (df['TargetProcessId'] == p)
]['TargetProcessGUID'].dropna()) - {NULL_GUID}

G = g1 | g2 | g3 | g4

print(f'G_1 = {g1}')
print(f'G_2 = {g2}')
print(f'G_3 = {g3}')
print(f'G_4 = {g4}')
print(f'\nG({p}, {c})')
print(f'  = {G}')
print(f'  |G| = {len(G)}')